In [1]:
import pandas as pd
import numpy as np

In [2]:
node_groups = pd.read_csv('../datasets/cloud_energy_consumption/node-groups/2024-12-14T000000Z_2025-04-13T235959Z/groups.csv')
node_groups

,node_group,model,form_factor,#sockets,cpu,#cores_per_socket,#threads_per_core,memory_type,memory_manufacturer,memory_used_slots,...,#infiniband_nics,infiniband_nic_model,#disks,disk_vendor,disk_model,disk_type,disk_size_GB,#fans,#power_supplies,rated_power
0,11cdff15,Fujitsu Server PRIMERGY BX924 S4,Blade Server,2,Intel(R) Xeon(R) CPU E5-2697 v2 @ 2.70GHz,12,2,DDR3,Hyundai Electronics (Hynix),1,...,0,-,1,Intel,INTEL_SSDSC2BA400G3C1,SSD,400.00,12 (per chassis),2 (per chassis),3200 W / 1600 W
1,b8994569,Lenovo ThinkSystem SD530,Chassis Server,2,Intel(R) Xeon(R) Gold 6252 CPU @ 2.10GHz,24,2,DDR4,Samsung,6,...,0,-,1,Samsung,MZ7LH480HAHQ-000V3_01PE095D7A09572LEN,SSD,480.00,10 (per 4 nodes),2 (per 4 nodes),2000 W
2,a97fe24e,Lenovo ThinkSystem SR630,Rack Server,2,Intel(R) Xeon(R) Gold 6230 CPU @ 2.10GHz,20,2,DDR4,Micron Technology,12,...,0,-,2,Lenovo,NaN,SSD,223.50,14,2,750 W
3,5f67cb23,Lenovo ThinkSystem SR650,Rack Server,2,Intel(R) Xeon(R) Gold 6230 CPU @ 2.10GHz,20,2,DDR4,SK Hynix,12,...,1,Mellanox MT27800 Family [ConnectX-5],2,Micron,MTFDDAK1T9TDC-1AT1ZA,SSD,1966.08,6,2,1600 W
4,f6fec747,Dell Inc. PowerEdge C6420,Chassis Server,2,Intel(R) Xeon(R) Gold 5220R CPU @ 2.20GHz,24,2,DDR4,002C069D002C,6,...,0,-,4,DELL,MZ7LH480HBHQ0D3,SSD,446.63,8,2 (per 4 nodes),2400 W
5,a6177608,Lenovo ThinkSystem SR630 V2,Rack Server,2,Intel(R) Xeon(R) Platinum 8358 CPU @ 2.60GHz,32,2,DDR4,Samsung,12,...,1,Mellanox MT28908 Family [ConnectX-6],2,Lenovo,MTFDDAV960TDS-1AW1ZA,M.2 SATA SSD,960.00,16,1,1100 W


## Rated Power

In [3]:
def parse_power(x):
    if '/' in str(x):
        total, usable = x.split('/')
        return float(total.strip().replace('W','')), float(usable.strip().replace('W',''))
    else:
        val = float(str(x).replace('W',''))
        return val, val

In [4]:
node_groups[['rated_power_total', 'rated_power_usable']] = (
    node_groups['rated_power'].apply(lambda x: pd.Series(parse_power(x)))
)

## CPU Frequency

In [8]:
node_groups['cpu_freq_ghz'] = node_groups['cpu'].str.extract(r'@ ([0-9.]+)GHz').astype(float)

## Cores

In [9]:
node_groups['total_cores'] = node_groups['#sockets'] * node_groups['#cores_per_socket']

## Threads

In [10]:
node_groups['total_threads'] = node_groups['total_cores'] * node_groups['#threads_per_core']

## Renaming

In [11]:
# changing the name of the columns for memory
node_groups.rename(columns={'memory_total_size_GB': 'memory_size_gb'}, inplace=True)
node_groups.rename(columns={'disk_size_GB': 'disk_size_gb'}, inplace=True)

## Form Factor (Blade, Chassis, Rack)
Form factor of the node. Can be either *"Rack Server"*, *"Chassis Server"* or *"Blade Server"*.

In [ ]:
pd.get_dummies(node_groups['form_factor'])
node_groups['is_blade'] = (node_groups['form_factor'] == 'Blade Server').astype(int)
node_groups['is_chassis'] = (node_groups['form_factor'] == 'Chassis Server').astype(int)
node_groups['is_rack'] = (node_groups['form_factor'] == 'Rack Server').astype(int)

## Dropping Noisy Columns / Keeping useful columns

In [14]:
keeping = [
    "node_group", 
    "form_factor",
    "total_cores", 
    "total_threads",
    "cpu_freq_ghz",
    "#gpus",
    "#disks",
    "#ethernet_nics",
    "#infiniband_nics",
    "disk_size_gb",
    "memory_size_gb",
    "memory_used_slots",
    "rated_power_total",
    "rated_power_usable"]

node_groups_final = node_groups[keeping]

In [15]:
node_groups_final

,node_group,form_factor,total_cores,total_threads,cpu_freq_ghz,#gpus,#disks,#ethernet_nics,#infiniband_nics,disk_size_gb,memory_size_gb,memory_used_slots,rated_power_total,rated_power_usable
0,11cdff15,Blade Server,24,48,2.7,0,1,2,0,400.00,16.0,1,3200.0,1600.0
1,b8994569,Chassis Server,48,96,2.1,0,1,2,0,480.00,192.0,6,2000.0,2000.0
2,a97fe24e,Rack Server,40,80,2.1,0,2,2,0,223.50,768.0,12,750.0,750.0
3,5f67cb23,Rack Server,40,80,2.1,2,2,2,1,1966.08,192.0,12,1600.0,1600.0
4,f6fec747,Chassis Server,48,96,2.2,0,4,2,0,446.63,192.0,6,2400.0,2400.0
5,a6177608,Rack Server,64,128,2.6,0,2,2,1,960.00,384.0,12,1100.0,1100.0


## Save

In [17]:
node_groups_final.to_csv('../datasets/cloud_energy_consumption/node-groups/2024-12-14T000000Z_2025-04-13T235959Z/cleaned_node_groups.csv', index=False)